# Lab 044 — NODE: build a differentiable oblivious tree, then race CatBoost

**Lesson:** [`lessons/0044-node.html`](../lessons/0044-node.html) · **Phase / Year:** Year 2 · Q1

**Paper:** Popov, Morozov & Babenko 2019, *Neural Oblivious Decision Ensembles for Deep Learning on Tabular Data* ([arXiv:1909.06312](https://arxiv.org/abs/1909.06312)) — §2 (architecture, Eq. 1–4) + Fig. 1 (DenseNet stacking). entmax: Peters, Niculae & Martins 2019 ([arXiv:1905.05702](https://arxiv.org/abs/1905.05702)). Oblivious/symmetric trees: CatBoost ([L016](../lessons/0016-catboost.html)).

**Dataset tier:** **A** — small real OpenML tables via `relkit` (CPU-cheap; deliberately NOT representative — see the #23 note in Task 3).

**Skill you are practising:** implement a differentiable oblivious tree **from scratch** — (1) `entmax15` by bisection, (2) the **oblivious tree forward pass** (entmoid split + outer-product routing) — then (3) race **CatBoost** (the same tree shape, grown greedily) under [L042](../lessons/0042-mlp-resnet-baselines.html)'s baseline-first rule, and (4) measure the cost.

**Exit criteria:** EXIT TICKET prints your entmax15 validation, the ODST forward match + leaf-routing check, the NODE-vs-CatBoost rank table, the cost ratio, and one sentence — *when is a differentiable tree actually worth it?*

---

### How this notebook works
- **PROVIDED** cells — boilerplate (data, frame, search spaces, CatBoost/net searches) **and** NODE (ODST, DenseNODE, train loop) copied into the notebook (not hidden behind `import relkit.node`); just run.
- **TODO** cells — blanks (`____`); you implement the skill.
- **CHECK** cells — immediate feedback; do not edit.
- Run top to bottom. After EXIT, a **NEXT STEP** cell trains closer to the paper (Colab GPU or Modal). When **EXIT TICKET** prints cleanly, paste it to your teacher or say *"lab done"*.

### Environment
One-time: `bash labs/setup-env.sh` → kernel **Relational Labs (.venv)**. Needs **torch** + scikit-learn + **catboost** (CPU is fine); the **`entmax`** package is used only to validate your entmax15. Real datasets fetch from OpenML on first run then cache. Budget: **~4–8 minutes on CPU** — set `OMP_NUM_THREADS=1` if a search feels slow (that has been the real cause of every slow lab so far). This lab uses a deliberately small budget/seed/dataset count to stay interactive; the lesson's headline numbers come from the fuller `labs/_verify_l044.py` run plus the paper's benchmark.

### Running on Google Colab?

Colab opens only this single file, so the course package (`relkit`) and the lab
dependencies (xgboost, lightgbm, catboost, …) are **not** present by default. The cell
below fixes that: on Colab it shallow-clones the course repo, installs
`requirements-labs.txt`, and switches into `labs/` so `relkit` imports and the data cache
resolve. **On a local venv or your own Jupyter it does nothing — just run it and continue.**

In [ ]:
# @colab-bootstrap — PROVIDED. Makes the lab self-sufficient on Google Colab; a no-op elsewhere.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/relational"):
        !git clone --depth 1 https://github.com/Avistian/relational.git /content/relational
    %pip install -q -r /content/relational/requirements-labs.txt
    os.chdir("/content/relational/labs")
    print("Colab ready — working dir:", os.getcwd())
else:
    print("Not on Colab — using the local environment as-is.")

## Concept recap — what NODE actually does

**The idea.** Take an **ensemble of oblivious decision trees** — the *symmetric* trees from [L016](../lessons/0016-catboost.html), where every node on a level shares one (feature, threshold), so a depth-`d` tree is a `2^d`-leaf lookup — and make the whole thing **differentiable**, so it trains end-to-end by gradient descent and stacks into deep layers like any neural net. A GBDT cannot do this: its greedy, discrete splits have no gradient, so it can never be a *layer* inside a larger learned model.

**The three softenings.**
1. **Feature choice → `entmax15`** (Task 1). Replace "pick feature j" with `f_hat = <x, entmax15(logits)>`, a learned sparse distribution over columns. `entmax15` (alpha = 1.5) is the middle of the family softmax (alpha→1, dense) — entmax15 (real zeros) — sparsemax (alpha = 2, sparsest, your L043 op), so the choice is genuinely selective yet differentiable.
2. **Split → `entmoid`** (Task 2). Replace the hard `1[f_hat > b]` with `c = entmoid((f_hat - b)/tau)`, the two-class entmax15. It saturates to an exact 0/1 for a decisive gap (a real decision) but smoothly, so a gradient flows; `tau` is the softness knob.
3. **Routing → outer product** (Task 2). A soft tree sends a *fraction* of the row to *every* leaf: leaf weight = product over levels of `c` or `1-c`. The outer product of the per-level `[c, 1-c]` gives all `2^d` leaf weights at once, summing to 1; the output is their weighted average of the leaf responses.

**NODE the network.** A NODE *layer* is an ensemble of hundreds of these trees; layers stack DenseNet-style (each sees the input plus earlier layers' outputs), and the prediction averages all trees. That stacking is the one thing a GBDT structurally cannot do.

**The honest verdict (Tasks 3–4).** On a single flat table this buys nothing: CatBoost — the same tree shape, grown greedily — ranks *above* NODE and trains ~70× faster. Differentiability pays off only when the tree must **compose** with other learned modules (embeddings, stacking, multi-modal) — which is the relational setting the thesis is about.

Full write-up + the interactive routing widget: [Lesson 044](../lessons/0044-node.html).

## Setup — PROVIDED (real tables + shared frame + search spaces)

In [ ]:
# PROVIDED — imports, the small real tables, the shared frame, and the search spaces.
# The NODE *architecture* is inlined in a later cell so you can read it; `relkit.node` here is only
# the Task-1/Task-2 *checker* (NOTES #22 / #25). Just run this cell.
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import torch
import torch.nn as nn
from scipy.stats import rankdata, friedmanchisquare
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))          # labs/ when run from there
sys.path.insert(0, str(Path(".").resolve().parent))   # labs/ when run from labs/solutions/
from relkit import load_tier_a
from relkit.nets import TabResNet, TabMLP, train_net, net_auc          # L042 baselines (from scratch)
from relkit.node import (entmax15 as relkit_entmax15,                  # checker only (NOTES #22)
                         entmoid15,                                    # Task 2 uses this operator
                         ODST as relkit_ODST)                          # checker for your forward

DEVICE = "cpu"
DATASETS = ["credit_g", "diabetes", "kc1"]   # small: NOT representative (see the #23 note below)
BUDGET, SEEDS = 3, [0, 1]                    # smaller than the lesson's run, so the lab stays interactive
EPOCHS, PATIENCE = 100, 10                   # same as labs/_verify_l044.py, so directions reproduce

def load_dense(name):
    Xdf, y = load_tier_a(name)
    num = Xdf.select_dtypes(include="number").columns.tolist()
    cat = [c for c in Xdf.columns if c not in num]
    ct = ColumnTransformer([("num", StandardScaler(), num),
                            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)])
    return ct.fit_transform(Xdf).astype(np.float32), y.to_numpy().astype(np.float32)

def frame(X, y, seed):
    """The SHARED frame — identical for every model (L020 contract, L042 protocol)."""
    Xtr_f, Xte, ytr_f, yte = train_test_split(X, y, test_size=0.30, random_state=seed, stratify=y)
    Xtr, Xva, ytr, yva = train_test_split(Xtr_f, ytr_f, test_size=0.25, random_state=seed,
                                          stratify=ytr_f)
    return Xtr, ytr, Xva, yva, Xte, yte

# ---- per-model search SPACES (equal BUDGET is what makes this fair — L038)
def sample_node(rng, d_in):
    return dict(in_features=d_in, num_trees=int(rng.choice([64, 128])),
                depth=int(rng.choice([3, 4, 5])), n_layers=int(rng.choice([1, 2]))), \
           dict(lr=float(rng.choice([0.005, 0.01, 0.02])))

def sample_net(kind, rng, d_in):
    opt = dict(lr=float(10 ** rng.uniform(-3.3, -2.5)), wd=float(10 ** rng.uniform(-6, -3)))
    if kind == "resnet":
        dm = int(rng.choice([64, 128, 192]))
        cfg = dict(d_in=d_in, d_main=dm, d_hidden=int(dm * rng.choice([1.0, 2.0])),
                   n_blocks=int(rng.choice([1, 2, 3])), dropout1=float(rng.uniform(0.0, 0.3)),
                   dropout2=float(rng.uniform(0.0, 0.3)))
    else:
        cfg = dict(d_in=d_in, d_block=int(rng.choice([64, 128, 256])),
                   n_blocks=int(rng.choice([1, 2, 3])), dropout=float(rng.uniform(0.0, 0.4)))
    return cfg, opt

def search_net(kind, Xtr, ytr, Xva, yva, Xte, yte, *, budget, seed):
    rng = np.random.default_rng(seed)
    best = {"val": -1.0, "test": None}
    for t in range(budget):
        cfg, opt = sample_net(kind, rng, Xtr.shape[1])
        m = TabResNet(**cfg) if kind == "resnet" else TabMLP(**cfg)
        m, val = train_net(m, Xtr, ytr, Xva, yva, lr=opt["lr"], wd=opt["wd"],
                           max_epochs=200, patience=16, seed=seed + t)
        if val > best["val"]:
            best = {"val": val, "test": net_auc(m, Xte, yte)}
    return best["test"]

def search_catboost(Xtr, ytr, Xva, yva, Xte, yte, *, budget, seed):
    """CatBoost = greedy oblivious (symmetric) trees (L016) — NODE's direct rival."""
    from catboost import CatBoostClassifier
    rng = np.random.default_rng(seed + 7)
    best = {"val": -1.0, "test": None}
    for _ in range(budget):
        clf = CatBoostClassifier(depth=int(rng.choice([4, 6, 8])),
                                 learning_rate=float(10 ** rng.uniform(-1.5, -0.7)),
                                 l2_leaf_reg=float(rng.choice([1.0, 3.0, 5.0, 9.0])),
                                 iterations=400, grow_policy="SymmetricTree", random_seed=seed,
                                 thread_count=2, verbose=0, allow_writing_files=False)
        clf.fit(Xtr, ytr.astype(int))
        val = roc_auc_score(yva, clf.predict_proba(Xva)[:, 1])
        if val > best["val"]:
            best = {"val": val, "test": roc_auc_score(yte, clf.predict_proba(Xte)[:, 1])}
    return best["test"]

print("setup ok — torch", torch.__version__)

## Task 1 — implement `entmax15` (the differentiable, sparse feature *choice*)

**Goal.** Write `entmax15`, the alpha = 1.5 member of the entmax family, by **bisection** on its threshold.
This is the operation NODE uses to *choose* which feature a tree level splits on.

**Why bisection.** Every alpha-entmax has the closed shape

```
p_i = [ (alpha - 1) * z_i - tau ]_+ ^ (1 / (alpha - 1))
```

for the unique `tau` that makes `p` sum to 1. There is no closed form for `tau` in general, but the sum
is monotone in `tau`, so we can **binary-search** it: pick a bracket `[tau_lo, tau_hi]`, and each step
halve it toward the `tau` whose `p` sums to exactly 1. Autograd differentiates straight through the
iterations, so no custom backward is needed.

**Why alpha = 1.5.** It is the sparse-but-gentle middle of the family: `softmax` (alpha->1) never zeros
anything, `sparsemax` (alpha=2, your L043 op) is the hardest, and `entmax15` sits between — real zeros,
smoother gradient. That is what NODE found trains best for feature selection.

In [ ]:
# TODO — implement entmax15 by bisection along the last dimension. Fill every ____.
def entmax15(z, n_iter=30):
    """z: (..., K) -> (..., K) alpha=1.5 entmax (a sparse distribution: sums to 1, exact zeros allowed)."""
    alpha = 1.5
    z = (alpha - 1) * z                        # fold (alpha-1) into z, so p_i = [z_i - tau]_+ ^ 2
    zmax = z.max(dim=-1, keepdim=True).values
    tau_lo = zmax - 1.0
    tau_hi = zmax - (1.0 / z.shape[-1]) ** (alpha - 1)
    for _ in range(n_iter):
        tau = (tau_lo + tau_hi) / 2
        # p from the current tau (clip negatives, then raise to 1/(alpha-1) = power 2)
        p = ____
        # if the mass is too small, tau is too high -> move the ceiling down; else move the floor up
        below = p.sum(dim=-1, keepdim=True) < 1.0
        tau_hi = torch.where(below, tau, tau_hi)
        tau_lo = torch.where(below, tau_lo, tau)
    tau = (tau_lo + tau_hi) / 2
    p = torch.clamp(z - tau, min=0) ** (1.0 / (alpha - 1))
    # normalise to kill tiny residual (bisection is approximate)
    return ____

zz = torch.tensor([[2.0, 1.5, 0.5, 0.0, -1.0]])
print("entmax15 :", np.round(entmax15(zz).numpy(), 3))
print("softmax  :", np.round(torch.softmax(zz, -1).numpy(), 3))

In [ ]:
# CHECK — the properties that make entmax15 a differentiable feature SELECTOR (do not edit)
rng = np.random.default_rng(0)
Z = torch.tensor(rng.normal(size=(200, 12)) * 3, dtype=torch.float64)
P = entmax15(Z, n_iter=50)

ok = True
def chk(name, cond, detail=""):
    global ok
    print(("PASS  " if cond else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))
    if not cond: ok = False

chk("sums to 1 along the feature axis", torch.allclose(P.sum(-1), torch.ones(200, dtype=torch.float64), atol=1e-6),
    f"max dev {(P.sum(-1) - 1).abs().max():.2e}")
chk("non-negative", bool((P >= 0).all()))
chk("contains EXACT zeros (softmax never does)",
    bool((P == 0).any()) and bool((torch.softmax(Z, -1) > 0).all()),
    f"{(P == 0).double().mean() * 100:.0f}% of entries are 0")
chk("sits BETWEEN softmax and sparsemax in sparsity",
    int((P == 0).sum()) > 0 and (P == 0).double().mean() < (Z.shape[1] - 1) / Z.shape[1])

# VALIDATION against the reference implementation (NOTES #22): the library checks you.
from entmax import entmax15 as ref_entmax15, sparsemax as ref_sparsemax
d = (P - ref_entmax15(Z, dim=-1)).abs().max().item()
chk("VALIDATED against entmax.entmax15", d < 1e-5, f"max |Δ| {d:.2e}")
chk("agrees with relkit.node.entmax15", torch.allclose(P, relkit_entmax15(Z, n_iter=50), atol=1e-9))

# entmax15 is denser than sparsemax on the same logits (fewer zeros)
zeros_ent = int((ref_entmax15(Z, dim=-1) == 0).sum())
zeros_spa = int((ref_sparsemax(Z, dim=-1) == 0).sum())
chk("entmax15 keeps MORE features alive than sparsemax", zeros_ent < zeros_spa,
    f"entmax zeros {zeros_ent} < sparsemax zeros {zeros_spa}")
print("\nTask 1", "OK" if ok else "-- fix the FAILs above")

## Task 2 — the oblivious tree forward pass (entmoid split + outer-product routing)

**Goal.** Implement the forward pass of one *ensemble of differentiable oblivious trees* (an `ODST` layer):
soft-split every level with the **entmoid**, then route each row to all `2^depth` leaves by an **outer
product**, and read off the weighted leaf responses.

The pieces, per (batch, tree, level):

```
f_hat = <x, entmax15(feature_logits)>            # the feature CHOICE (Task 1), one per level
c     = entmoid( (f_hat - threshold) / exp(log_temp) )   # the soft SPLIT, in [0, 1]
bins  = stack([c, 1 - c])                         # go-right / go-left probabilities
w     = product over levels of the matching bin   # leaf ROUTING weights, 2^depth of them, sum to 1
out   = sum over leaves of  w * response           # weighted average of the leaf responses
```

**Why this matters.** This forward pass is the whole trick: because every step is a smooth function
(entmax choice, entmoid split, product routing), the 2^depth-leaf tree is differentiable and trains by
backprop. The `entmoid` is just the two-class entmax15 — it saturates to an exact 0/1 for a decisive gap
(a real decision) but stays smooth, so a gradient flows. You will confirm your forward matches the
from-scratch reference `relkit_ODST` exactly when they share parameters.

In [ ]:
# TODO — fill every ____.  (entmax15 from Task 1; entmoid15 provided by relkit.)
def odst_forward(x, feature_logits, thresholds, log_temperatures, response, bin_codes_1hot):
    """One ensemble of oblivious trees. Shapes:
       x:(B,in)  feature_logits:(in,T,d)  thresholds:(T,d)  log_temperatures:(T,d)
       response:(T,tree_dim,2^d)  bin_codes_1hot:(d,2^d,2)   ->  (B, T*tree_dim)
    """
    B = x.shape[0]
    feature_selectors = entmax15(feature_logits.movedim(0, -1)).movedim(-1, 0)   # entmax over features
    f_hat = torch.einsum("bi,itl->btl", x, feature_selectors)                    # (B,T,d) chosen values

    # the soft split: scale the gap by 1/temperature, then entmoid -> P(go right) in [0,1]
    c = ____
    bins = torch.stack([c, 1 - c], dim=-1)                                       # (B,T,d,2)

    # route to leaves: match each leaf's per-level bit, then take the PRODUCT over levels
    bin_matches = torch.einsum("btds,dls->btdl", bins, bin_codes_1hot)           # (B,T,d,2^d)
    weights = ____                                                               # (B,T,2^d), sums to 1

    # weighted average of leaf responses
    out = torch.einsum("btl,tcl->btc", weights, response)                        # (B,T,tree_dim)
    return out.reshape(B, -1)

# Build a reference ODST, then run YOUR forward on ITS parameters and compare (NOTES #22).
torch.manual_seed(0)
ref = relkit_ODST(in_features=10, num_trees=8, depth=4, tree_dim=1).eval()
x = torch.randn(32, 10)
mine = odst_forward(x, ref.feature_logits, ref.thresholds, ref.log_temperatures,
                    ref.response, ref.bin_codes_1hot)
with torch.no_grad():
    theirs = ref(x)
print("max |Δ| vs reference ODST:", (mine - theirs).abs().max().item())

In [ ]:
# CHECK — the invariants of the differentiable oblivious tree (do not edit)
ok = True
def chk(name, cond, detail=""):
    global ok
    print(("PASS  " if cond else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))
    if not cond: ok = False

# recompute the routing weights to inspect them directly
fs = entmax15(ref.feature_logits.movedim(0, -1)).movedim(-1, 0)
f_hat = torch.einsum("bi,itl->btl", x, fs)
c = entmoid15((f_hat - ref.thresholds) * torch.exp(-ref.log_temperatures))
bins = torch.stack([c, 1 - c], dim=-1)
weights = torch.einsum("btds,dls->btdl", bins, ref.bin_codes_1hot).prod(dim=-2)

chk("forward MATCHES the reference ODST on shared parameters",
    torch.allclose(mine, ref(x), atol=1e-5), f"max |Δ| {(mine - ref(x)).abs().max():.2e}")
chk("leaf-routing weights sum to 1 per (row, tree)",
    torch.allclose(weights.sum(-1), torch.ones(32, 8), atol=1e-4),
    f"max dev {(weights.sum(-1) - 1).abs().max():.2e}")
chk("there are 2^depth = 16 leaves", weights.shape[-1] == 16)
chk("entmoid gives a genuine hard decision for a decisive gap (exact 0/1)",
    float(entmoid15(torch.tensor([9.0]))) == 1.0 and float(entmoid15(torch.tensor([-9.0]))) == 0.0)

# temperature -> 0 collapses the soft routing onto ONE leaf (a hard tree)
c_hard = entmoid15((f_hat - ref.thresholds) / 0.01)
w_hard = torch.stack([c_hard, 1 - c_hard], -1)
w_hard = torch.einsum("btds,dls->btdl", w_hard, ref.bin_codes_1hot).prod(dim=-2)
chk("tau -> 0 makes routing ~one-hot (max leaf weight > 0.98)", float(w_hard.max(-1).values.mean()) > 0.98,
    f"mean max-leaf weight {float(w_hard.max(-1).values.mean()):.3f}")
print("\nTask 2", "OK" if ok else "-- fix the FAILs above")

In [ ]:
# PROVIDED — adapter: your Task-1 entmax15 is last-axis only; the inlined ODST calls dim=0.
# Bind `_impl` at def-time (and stash it) so re-running this cell cannot wrap the wrapper —
# that RecursionError's on the next train, not here.
_impl = getattr(entmax15, "_impl", entmax15)
def entmax15(z, dim=-1, n_iter=30, *, _impl=_impl):
    """Keep YOUR Task-1 implementation; accept the encoder's dim= keyword."""
    if dim in (-1, z.ndim - 1):
        return _impl(z, n_iter=n_iter)
    z_t = z.transpose(dim, -1)
    return _impl(z_t, n_iter=n_iter).transpose(dim, -1)
entmax15._impl = _impl


## The rest of the paper's architecture — inlined, not imported

The next cell is `labs/relkit/node.py` copied into this notebook so you can **read every line** of ODST, DenseNODE, and the training loop. It is not `from relkit import ...` hiding a model behind a package. `entmax15` defined in your TODO cells above are **kept** — this copy skips those names, so the encoder you train next calls *your* functions.

`relkit/` still holds the canonical file for Modal / `_verify` so the two cannot drift in opposite directions; the notebook is a readable copy, not a black box.

In [ ]:
# PROVIDED — inlined from `labs/relkit/node.py` so you can read every line.
# This is the paper implementation, not `import relkit...` hiding it.
# Canonical file stays at labs/relkit/node.py for Modal / _verify; this cell is a copy.

"""From-scratch NODE (Popov, Morozov & Babenko 2019, arXiv:1909.06312) — Lesson 044.

Neural Oblivious Decision Ensembles. Built from the paper, not from a library (NOTES standards
#18/#22/#24). The `entmax` package (Peters et al. 2019) is used ONLY as a VALIDATION point in
`labs/_verify_l044.py` and `labs/_check_l044.py` — never imported here.

Paper map — every piece below cites the element it realises (paper §2 "Neural Oblivious Decision
Ensembles", Eq. 1–4, Fig. 1):

| Paper                                                        | Here                        |
|-------------------------------------------------------------|-----------------------------|
| alpha-entmax feature choice  F_hat_i = <x, entmax_a(F_i)>    | `entmax15` + `ODST.forward` |
| two-class entmax split (the "entmoid")  c_i in [0,1]         | `entmoid15`                 |
| choice tensor  C(x) = outer_i [c_i, 1-c_i]  (2^d leaves)     | `ODST.forward` (`weights`)  |
| tree output  h(x) = <C(x), R>                                | `ODST.forward` (`response`) |
| oblivious tree: ONE (feature, threshold) shared per level    | `ODST` parameter shapes     |
| DenseNet-style multi-layer stacking (Fig. 1)                | `DenseNODE`                 |

Binary classification only (tree_dim = 1, averaged over trees to one logit) — all Lesson 044 needs.
CPU is fine for the small Tier-A tables.

Why a differentiable oblivious tree at all? An oblivious decision tree (CatBoost's symmetric tree,
L016) uses the SAME split feature + threshold at every node of a level, so a depth-`d` tree is fully
described by `d` features, `d` thresholds, and 2^d leaf responses. That regularity is what lets us make
it differentiable: replace "pick feature j" with a sparse `entmax` choice over features, and replace the
hard comparison `f > b` with a soft, temperature-controlled `entmoid`. Every leaf then receives a
*fraction* of the row (they sum to 1), so the whole tree is a smooth function of its parameters and
trains by gradient descent end-to-end — unlike a GBDT, whose splits are chosen greedily and are not
differentiable.
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score


# ---------------------------------------------------------------------------- sparse transforms
def entmoid15(x: torch.Tensor) -> torch.Tensor:
    """Two-class alpha=1.5 entmax — the "entmoid", NODE's differentiable, sparse replacement for sigmoid.

    entmoid15(x) is exactly `entmax15([x, 0])[..., 0]`: the probability the soft split sends a row to
    the "greater-than-threshold" side. Because it is entmax, once `|x|` is large enough the row is routed
    fully (probability an exact 0 or 1) — a genuinely discrete decision, reached smoothly. We use the
    closed form (NODE repo `lib/nn_utils.py`), which _check_l044 asserts equals the two-class entmax.

        tau = (|x| + sqrt(relu(8 - x^2))) / 2 ;  y = 0.25 * relu(tau - |x|)^2  (the smaller-class mass)
    """
    is_pos = x >= 0
    ax = x.abs()
    tau = (ax + torch.sqrt(F.relu(8 - ax ** 2))) / 2
    tau = torch.where(tau <= ax, torch.full_like(tau, 2.0), tau)
    y_neg = 0.25 * F.relu(tau - ax) ** 2
    return torch.where(is_pos, 1 - y_neg, y_neg)


# ---------------------------------------------------------------------------- one NODE layer (ODST)
class ODST(nn.Module):
    """An ensemble of `num_trees` differentiable Oblivious Decision Trees of depth `depth` (paper §2).

    "Oblivious" = the tree uses ONE (feature, threshold) pair per LEVEL, shared by every node in that
    level (CatBoost's symmetric tree, L016). So the parameters are just:

      feature_logits  F : [in_features, num_trees, depth]   -> a sparse feature choice per (tree, level)
      thresholds      b : [num_trees, depth]                 -> one split point per (tree, level)
      log_temp    log_t : [num_trees, depth]                 -> soft-split temperature per (tree, level)
      response        R : [num_trees, tree_dim, 2^depth]     -> a learnable answer per leaf

    Forward (Eq. 2–4):
      1. feature choice   f_hat[b,t,l] = <x[b], entmax15(F[:,t,l])>            (differentiable "pick")
      2. soft split       c[b,t,l]     = entmoid15( (f_hat - b) * exp(-log_t) )
      3. routing tensor   w[b,t,leaf]  = prod_l  ( c or 1-c, per the leaf's bit at level l )
      4. tree output      out[b,t,:]   = sum_leaf w * R                        (weighted leaf average)

    Returns [batch, num_trees * tree_dim].
    """

    def __init__(self, in_features, num_trees=128, depth=6, tree_dim=1):
        super().__init__()
        self.in_features, self.num_trees, self.depth, self.tree_dim = in_features, num_trees, depth, tree_dim

        self.feature_logits = nn.Parameter(torch.randn(in_features, num_trees, depth))
        self.thresholds = nn.Parameter(torch.randn(num_trees, depth))
        self.log_temperatures = nn.Parameter(torch.zeros(num_trees, depth))
        self.response = nn.Parameter(torch.randn(num_trees, tree_dim, 2 ** depth) * 0.1)

        # bin_codes_1hot[l, leaf, s]: at level l, which side (s=0 "right"/c, s=1 "left"/1-c) leaf uses.
        leaves = torch.arange(2 ** depth)
        offsets = 2 ** torch.arange(depth)
        bit = (leaves.view(1, -1) // offsets.view(-1, 1)) % 2          # [depth, 2^depth], 1 => "right"
        bin_codes = torch.stack([bit, 1 - bit], dim=-1).float()        # [depth, 2^depth, 2]
        self.register_buffer("bin_codes_1hot", bin_codes)

    def initialize(self, x: torch.Tensor):
        """Data-aware init (paper App.): set thresholds to sampled feature quantiles so early splits are
        informative, and scale temperatures to the spread of the chosen feature values."""
        with torch.no_grad():
            fs = entmax15(self.feature_logits, dim=0)                  # [in, trees, depth]
            f_hat = torch.einsum("bi,itl->btl", x, fs)                 # [batch, trees, depth]
            # thresholds <- a random row's feature value per (tree, level); temperature <- its std.
            idx = torch.randint(0, x.shape[0], (self.num_trees, self.depth))
            self.thresholds.data = f_hat[idx, torch.arange(self.num_trees).view(-1, 1),
                                         torch.arange(self.depth).view(1, -1)]
            self.log_temperatures.data = torch.log(f_hat.std(dim=0).clamp_min(1e-2))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feature_selectors = entmax15(self.feature_logits, dim=0)       # sparse choice over features
        f_hat = torch.einsum("bi,itl->btl", x, feature_selectors)      # [batch, trees, depth]
        logits = (f_hat - self.thresholds) * torch.exp(-self.log_temperatures)
        c = entmoid15(logits)                                          # [batch, trees, depth] soft split
        # Paper: C(x) = outer_l [c_l, 1-c_l] over depth (Eq. 3). Built iteratively so the
        # peak tensor is [batch, trees, 2^depth], not [batch, trees, depth, 2^depth] —
        # the latter is ~11 GB on a Higgs test split and OOMs a T4 at paper HPs (2048 trees).
        # Bit order matches `bin_codes_1hot`: level ℓ is the 2^ℓ place (leaf += 2^ℓ if "right"/c).
        weights = x.new_ones(x.shape[0], self.num_trees, 1)
        for level in range(self.depth):
            cl = c[:, :, level].unsqueeze(-1)
            weights = torch.cat([weights * (1.0 - cl), weights * cl], dim=-1)
        out = torch.einsum("btl,tcl->btc", weights, self.response)     # [batch, trees, tree_dim]
        return out.reshape(x.shape[0], self.num_trees * self.tree_dim)


class DenseNODE(nn.Module):
    """NODE = one or more ODST layers, DenseNet-style: each layer sees the input PLUS every earlier
    layer's tree outputs (Fig. 1). The prediction averages the first output unit of every tree across
    all layers (paper: average of all trees' responses), giving one logit for binary classification.
    """

    def __init__(self, in_features, num_trees=128, depth=6, n_layers=1, tree_dim=1):
        super().__init__()
        self.n_layers, self.num_trees, self.tree_dim = n_layers, num_trees, tree_dim
        self.layers = nn.ModuleList()
        d = in_features
        for _ in range(n_layers):
            self.layers.append(ODST(d, num_trees=num_trees, depth=depth, tree_dim=tree_dim))
            d += num_trees * tree_dim                                  # dense concat of this layer's out

    @torch.no_grad()
    def initialize(self, x):
        h = x
        for layer in self.layers:
            layer.initialize(h)
            h = torch.cat([h, layer(h)], dim=-1)

    def forward(self, x):
        h = x
        outputs = []
        for layer in self.layers:
            out = layer(h)                                             # [batch, trees*tree_dim]
            outputs.append(out.view(x.shape[0], self.num_trees, self.tree_dim)[..., 0])
            h = torch.cat([h, out], dim=-1)
        # average the first response unit over all trees in all layers -> one logit
        return torch.cat(outputs, dim=1).mean(dim=1)


def _forward_in_batches(model, x: torch.Tensor, batch_size: int) -> torch.Tensor:
    """Chunked forward so ODST's `[N, trees, depth, 2^depth]` routing tensor fits in memory.

    A single Higgs test forward at the closer preset (N≈29k, 256 trees, depth 6) is ~11 GB
    for that tensor alone and OOMs a Colab T4; training batches of 512 are fine.
    """
    n = x.size(0)
    bs = max(1, min(int(batch_size), n))
    if n <= bs:
        return model(x)
    return torch.cat([model(x[s:s + bs]) for s in range(0, n, bs)], dim=0)


def train_node(model, Xtr, ytr, Xva, yva, *, lr=1e-3, wd=0.0, max_epochs=100, patience=10,
               batch_size=512, device="cpu", seed=0):
    """Mini-batch Adam with early stopping on validation ROC-AUC — the same fair, shared-protocol
    contract as `relkit.nets.train_net` (L042) and `relkit.tabnet.train_tabnet` (L043): every model
    picks its own training length by validation, so no arm is accidentally under- or over-trained.

    NODE-specific: a data-aware `initialize` pass on the first training batch (paper appendix) so the
    thresholds start at real feature quantiles rather than random N(0,1) draws.

    Returns (model_with_best_val_weights, best_val_auc).
    """
    torch.manual_seed(seed)
    model = model.to(device)
    Xt = torch.tensor(np.asarray(Xtr), dtype=torch.float32, device=device)
    yt = torch.tensor(np.asarray(ytr), dtype=torch.float32, device=device)
    Xv = torch.tensor(np.asarray(Xva), dtype=torch.float32, device=device)
    n = Xt.size(0)
    bs = min(batch_size, n)

    model.initialize(Xt[:min(n, bs)])                                  # data-aware threshold init
    if device == "cuda":
        torch.cuda.empty_cache()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    lossf = nn.BCEWithLogitsLoss()
    generator = torch.Generator().manual_seed(seed)

    best_auc, best_state, since = -1.0, None, 0
    for _ in range(max_epochs):
        model.train()
        perm = torch.randperm(n, generator=generator)
        for start in range(0, n, bs):
            idx = perm[start:start + bs]
            if idx.numel() < 2:
                continue
            opt.zero_grad()
            loss = lossf(model(Xt[idx]), yt[idx])
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            pv = torch.sigmoid(_forward_in_batches(model, Xv, bs)).cpu().numpy()
        val_auc = roc_auc_score(yva, pv)
        if val_auc > best_auc:
            best_auc, since = val_auc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            since += 1
            if since >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_auc


@torch.no_grad()
def node_predict_logits(model, X, *, device="cpu", batch_size=512):
    """Logits for `X`, evaluated in batches of `batch_size` (see `_forward_in_batches`)."""
    model.eval()
    xt = torch.as_tensor(np.asarray(X), dtype=torch.float32, device=device)
    return _forward_in_batches(model, xt, batch_size)


@torch.no_grad()
def node_auc(model, X, y, *, device="cpu", batch_size=512):
    logits = node_predict_logits(model, X, device=device, batch_size=batch_size)
    return roc_auc_score(y, torch.sigmoid(logits).cpu().numpy())


## Task 3 — race CatBoost (the clean head-to-head) and the L042 baselines

**Goal.** Run NODE against **CatBoost** and the tuned MLP/ResNet under one shared frame on several tables,
then summarise with **mean ranks** and a Friedman test.

**Why CatBoost specifically.** NODE and CatBoost are the *same tree shape* — ensembles of oblivious
(symmetric) trees — differing only in how splits are chosen (NODE by gradient descent, CatBoost greedily).
So this isolates the paper's central question: does making the oblivious tree differentiable buy accuracy?
And it is [L042](../lessons/0042-mlp-resnet-baselines.html)'s baseline-first rule: the cheap strong models
are run *first*, to the same protocol, and the expensive new model must beat them to earn its place.

**And why several datasets (NOTES #23).** A single table is a demonstration, never evidence. This set is
small and deliberately *not* representative — the strong claim stays cited to the paper's own 40+ dataset
study (which used far more trees and tuning than this budget).

In [ ]:
# TODO — fill every ____.
def search_node(Xtr, ytr, Xva, yva, Xte, yte, *, budget, seed):
    """Validation-selected random search over NODE's space — SAME budget as every other model."""
    rng = np.random.default_rng(seed)
    best = {"val": -1.0, "test": None}
    for t in range(budget):
        cfg, opt = sample_node(rng, Xtr.shape[1])
        torch.manual_seed(seed + t)
        model, val = train_node(DenseNODE(**cfg), Xtr, ytr, Xva, yva, lr=opt["lr"],
                                max_epochs=EPOCHS, patience=PATIENCE, seed=seed + t)
        # keep the trial with the best VALIDATION score, and score IT on test (never select on test)
        if ____:
            best = {"val": val, "test": ____}
    return best["test"]

MODELS = ["node", "catboost", "mlp", "resnet"]
table = {}
for name in DATASETS:
    X, y = load_dense(name)
    rows = {m: [] for m in MODELS}
    for s in SEEDS:
        f = frame(X, y, s)
        rows["node"].append(search_node(*f, budget=BUDGET, seed=s))
        rows["catboost"].append(search_catboost(*f, budget=BUDGET, seed=s))
        rows["mlp"].append(search_net("mlp", *f, budget=BUDGET, seed=s))
        rows["resnet"].append(search_net("resnet", *f, budget=BUDGET, seed=s))
    table[name] = {m: (float(np.mean(v)), float(np.std(v))) for m, v in rows.items()}
    print(f"{name:>10}: " + " | ".join(f"{m} {table[name][m][0]:.3f}±{table[name][m][1]:.3f}"
                                       for m in MODELS), flush=True)

# --- cross-dataset summary: mean ranks (1 = best per dataset) + Friedman
score = np.array([[table[d][m][0] for m in MODELS] for d in DATASETS])   # datasets x models
ranks = np.array([rankdata(-row, method="average") for row in score])    # HIGHEST score -> rank 1
mean_rank = {m: float(ranks[:, i].mean()) for i, m in enumerate(MODELS)}
fried = friedmanchisquare(*[score[:, i] for i in range(len(MODELS))])
node_beats_cat = sum(table[d]["node"][0] > table[d]["catboost"][0] for d in DATASETS)

print("\nmean ranks:", {m: round(r, 2) for m, r in mean_rank.items()})
print(f"Friedman chi2={fried.statistic:.3f}, p={fried.pvalue:.3f} "
      f"(k={len(MODELS)}, N={len(DATASETS)})")
print(f"NODE beats CatBoost on {node_beats_cat}/{len(DATASETS)} tables")

In [ ]:
# CHECK — the verdict, read the disciplined way (do not edit)
ok = True
def chk(name, cond, detail=""):
    global ok
    print(("PASS  " if cond else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))
    if not cond: ok = False

chk("ranks are a valid per-dataset ranking of the 4 models",
    ranks.shape == (len(DATASETS), 4) and bool(np.allclose(ranks.sum(1), 10.0)))
chk("best score on each dataset got rank 1",
    all(ranks[i][np.argmax(score[i])] == 1 for i in range(len(DATASETS))))
chk("every model was given the SAME budget", True, f"budget={BUDGET} per model, seeds={SEEDS}")

cat_rank = mean_rank["catboost"]
print(f"""
VERDICT — NODE mean rank {mean_rank['node']:.2f} vs CatBoost {cat_rank:.2f} (its own tree shape, grown greedily)
  baseline-first outcome : {'CLEARED the bar' if mean_rank['node'] < min(mean_rank['catboost'], mean_rank['mlp'], mean_rank['resnet']) else 'did NOT clear the bar'}
  Friedman p             : {fried.pvalue:.3f} -> {'a difference is detectable' if fried.pvalue < 0.05 else 'CANNOT distinguish these models on this sample'}

Keep the two statements separate. p > 0.05 means you may NOT claim NODE is *significantly worse* — this
sample is far too small. But the burden of proof sits with the NEW, more expensive model, so ranking
behind the CatBoost it generalises means it did not clear the bar HERE. With a small budget and {len(SEEDS)}
seeds your exact numbers will differ from the lesson's fuller run (labs/_verify_l044.py: NODE 3.50,
CatBoost 2.50, MLP 2.00, ResNet 2.00, p = 0.308) — what should reproduce is the DIRECTION. And this is a
down-scaled experiment: NODE's paper win is at benchmark scale with thousands of trees, not this budget.""")
print("Task 3", "OK" if ok else "-- fix the FAILs above")

## Task 4 — the other axis: what does the differentiability cost?

**Goal.** Time NODE against the CatBoost it imitates on one table, so your verdict weighs accuracy *and*
compute — the honest full picture.

**Why.** Accuracy alone hides half the story. NODE and CatBoost are the same tree shape, but NODE trains
every split by backprop over many epochs while CatBoost grows them greedily in one pass. Measuring the
wall-clock gap is what turns "NODE is a bit behind" into "NODE is a bit behind *and* far more expensive" —
which is the point of the lesson.

In [ ]:
# TODO — fill every ____.
from catboost import CatBoostClassifier
X, y = load_dense("credit_g")
Xtr, ytr, Xva, yva, Xte, yte = frame(X, y, 0)

# time NODE (128 trees, depth 6)
t0 = time.time(); torch.manual_seed(0)
node = DenseNODE(X.shape[1], num_trees=128, depth=6, n_layers=1)
node, _ = train_node(node, Xtr, ytr, Xva, yva, lr=1e-2, max_epochs=EPOCHS, patience=PATIENCE, seed=0)
node_s = time.time() - t0
node_test = node_auc(node, Xte, yte)

# time CatBoost (400 symmetric trees) on the SAME split
t0 = time.time()
cat = CatBoostClassifier(depth=6, iterations=400, learning_rate=0.05, grow_policy="SymmetricTree",
                         random_seed=0, thread_count=2, verbose=0, allow_writing_files=False)
cat.fit(Xtr, ytr.astype(int))
cat_s = time.time() - t0
cat_test = roc_auc_score(yte, cat.predict_proba(Xte)[:, 1])

# the slowdown factor
slowdown = ____

print(f"NODE     : {node_s:6.1f}s   test AUC {node_test:.3f}")
print(f"CatBoost : {cat_s:6.1f}s   test AUC {cat_test:.3f}")
print(f"NODE is {slowdown:.0f}x slower to train on this table")

In [ ]:
# CHECK — cost is real and belongs in the verdict (do not edit)
ok = True
def chk(name, cond, detail=""):
    global ok
    print(("PASS  " if cond else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))
    if not cond: ok = False

chk("NODE is materially slower than CatBoost (>= 5x)", slowdown >= 5, f"{slowdown:.0f}x")
chk("both produce a valid AUC", 0.5 < node_test < 1.0 and 0.5 < cat_test < 1.0)
print(f"""
The lesson's measured run: NODE 60.2 s vs CatBoost 0.9 s (~70x), NODE 0.793 vs CatBoost 0.813 AUC. Your
absolute times depend on your CPU, but the SHAPE holds: the differentiable tree costs far more and, on one
flat table, buys nothing on the metric. That is the whole trade — differentiability is worth paying for
only when the tree must COMPOSE with other learned modules (joint embeddings, DenseNet stacking,
end-to-end multi-modal), which is exactly the relational setting the thesis cares about.""")
print("Task 4", "OK" if ok else "-- fix the FAILs above")

## EXIT TICKET

Paste this output to your teacher, or just say *"lab done."*

In [ ]:
# EXIT TICKET
print("=== LAB 044 — NODE (differentiable oblivious trees) ===")
print(f"entmax15         : validated (exact zeros, between softmax & sparsemax, matches reference)")
print(f"oblivious tree   : forward matches relkit.node.ODST; leaf routing sums to 1; tau->0 => one leaf")
print(f"bake-off ranks   : " + ", ".join(f"{m} {mean_rank[m]:.2f}" for m in MODELS))
print(f"Friedman         : chi2={fried.statistic:.3f}, p={fried.pvalue:.3f} (N={len(DATASETS)} datasets)")
print(f"NODE vs CatBoost : NODE wins {node_beats_cat}/{len(DATASETS)} tables")
print(f"cost             : NODE {node_s:.1f}s vs CatBoost {cat_s:.1f}s ({slowdown:.0f}x slower)")
print(f"cleared the bar? : {'YES' if mean_rank['node'] < min(mean_rank['catboost'], mean_rank['mlp'], mean_rank['resnet']) else 'NO'}")
print()
print("when would you reach for a differentiable tree over CatBoost?:", "____")

## NEXT STEP — reproduce the paper's results (required, not stretch)

The EXIT ticket above is the **learning lab**: you implemented the architecture and ran a
downscaled bake-off that fits in minutes on CPU. That is a *different experiment* from the
paper's table. **Do not treat the EXIT ranking as the paper's result.** Mixing those two
buckets is how you learn the wrong conclusion (standard #25 / M60).

**Paper:** Popov, Morozov & Babenko 2020, Neural Oblivious Decision Ensembles ([arXiv:1909.06312](https://arxiv.org/abs/1909.06312))

**Bucket 1 — verified here (this notebook's budget)**
- **mean ranks:** NODE 3.50 vs CatBoost 2.50 / MLP 2.00 / ResNet 2.00, Friedman p=0.308
- **cost:** ~70× slower than CatBoost on credit_g at this budget

**Bucket 2 — the paper's claim (cited, not yet reproduced by this notebook)**
- **Higgs default-HP error:** NODE 0.2412 vs CatBoost 0.2434 (Table 1). We use OpenML 23512 (~98k of 10.5M) → expect INCOMPARABLE on the absolute number; read DIRECTION.

**Bucket 3 — scale-up run (you train this).** Same from-scratch code, closer to the paper's
dataset / budget / metric. Two operators:

1. **Google Colab (you, GPU).** `Runtime → Change runtime type → T4 GPU`, set
   `RUN_PAPER_REPRO = True` in the next cell, run it. Stay in the tab — free Colab
   disconnects after ~90 min of no *tab* interaction, even if training is still going.
2. **Modal (unattended).** From the repo root:
   ```
   ~/.local/bin/modal run --detach modal/l044_paper_repro.py --preset closer
   ```
   Use `--preset paper` only when you can spend hours and want paper hyperparameters.
   `smoke` is a seconds-long import check, not a result.

When it finishes, the cell prints a **ledger** with MATCH / CLOSE / FAIL / INCOMPARABLE /
DIRECTION_*. Paste that ledger to your teacher. Until you run it, the paper claim stays
*cited, not reproduced* — and that is an honest state, not a failure.


In [ ]:
# PROVIDED — paper-results scale-up, inlined from `labs/_paper_repro_l044.py`.
# Read this cell: it is the training / comparison loop, not a hidden package.
# Default OFF so the learning lab stays minutes. On Colab: Runtime → T4 GPU,
# set RUN_PAPER_REPRO = True, re-run. Unattended: modal run --detach modal/l044_paper_repro.py

"""L044 paper-results scale-up (NOTES standard #25).

The learning lab races NODE vs CatBoost on four *small* tables with ~64–128 trees.
Popov, Morozov & Babenko 2020 (NODE) claim a *default-HP* win over CatBoost/XGBoost on
six large datasets with **2048 trees of depth 6**. That is a different experiment.

This harness trains the from-scratch NODE against CatBoost on a documented subsample of
the paper's Higgs table (OpenML 23512, ~98k of the paper's 10.5M) — or falls back to
Adult with an honest gap. The paper's Higgs default-HP gap is ~0.002 error (NODE 0.2412
vs CatBoost 0.2434). A 98k subsample will often be a DIRECTION_TIE; that is a lesson
about *statistical power*, not a refutation of Table 1.

Presets: smoke · closer · paper.

Run:
    OMP_NUM_THREADS=1 python labs/_paper_repro_l044.py --preset smoke
    ~/.local/bin/modal run --detach modal/l044_paper_repro.py --preset closer
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
import traceback
import warnings

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
sys.path.insert(0, HERE)

from relkit.paper_repro import (  # noqa: E402
    LabFinding, PaperTarget, ScaleUpRun,
    classify_direction, classify_number, device, format_ledger, hardware_tag,
    print_howto, to_jsonable,
)

HIGGS_TARGET = PaperTarget(
    paper="NODE (Popov, Morozov & Babenko 2020)", arxiv="1909.06312",
    table="Table 1 — Higgs, default HPs (classification error)",
    dataset="Higgs", metric="error", paper_value=0.2412, paper_std=0.0005, abs_tol=0.005,
    paper_split="paper: 10.5M train / 500k test; we use OpenML 23512 (~98k) or Adult fallback",
    higher_is_better=False,
    notes="Default NODE = 1 layer × 2048 trees × depth 6. CatBoost Table 1 Higgs error = 0.2434.",
)

LAB_FINDINGS = [
    LabFinding("NODE vs CatBoost/MLP/ResNet mean ranks",
               "NODE 3.50 · CatBoost 2.50 · MLP 2.00 · ResNet 2.00 (Friedman p=0.308); "
               "NODE beats CatBoost 1/4; ~70× slower on credit_g",
               "4 small OpenML tables, 64–128 trees, 3 seeds, CPU — NOT the paper's 40+/6-dataset study"),
]


def _dense_from_xy(Xdf, y):
    # OpenML 23512 (higgs_small) has one incomplete row. StandardScaler leaves those
    # NaNs in place; NODE then emits NaN logits and sklearn raises "Input contains NaN."
    keep = Xdf.notna().all(axis=1)
    if not bool(keep.all()):
        Xdf = Xdf.loc[keep]
        y = y.loc[keep] if hasattr(y, "loc") else np.asarray(y)[np.asarray(keep)]
    num = Xdf.select_dtypes(include="number").columns.tolist()
    cat = [c for c in Xdf.columns if c not in num]
    ct = ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
    ])
    return ct.fit_transform(Xdf).astype(np.float32), np.asarray(y).astype(np.float32)


def load_table():
    """Prefer the paper's Higgs (subsampled); fall back to Adult with a gap note."""
    from relkit import load_tier_a
    try:
        Xdf, y = load_tier_a("higgs_small")
        gaps = [
            "OpenML 23512 (~98k rows) is a SUBSAMPLE of UCI Higgs; paper used 10.5M / 500k test",
        ]
        name = "higgs_small"
    except Exception as exc:
        Xdf, y = load_tier_a("adult")
        gaps = [
            f"higgs_small failed ({exc}); Adult is NOT a NODE paper dataset — Table 1 number is INCOMPARABLE",
        ]
        name = "adult"
    n_incomplete = int(Xdf.isna().any(axis=1).sum())
    if n_incomplete:
        gaps = list(gaps) + [
            f"dropped {n_incomplete} incomplete OpenML row(s) with NaN features",
        ]
    return _dense_from_xy(Xdf, y), name, gaps


def _split(X, y, seed):
    Xtr_f, Xte, ytr_f, yte = train_test_split(X, y, test_size=0.30, random_state=seed, stratify=y)
    Xtr, Xva, ytr, yva = train_test_split(Xtr_f, ytr_f, test_size=0.25, random_state=seed, stratify=ytr_f)
    return Xtr, ytr, Xva, yva, Xte, yte


def _error_auc(predict_proba_pos, y):
    pred = (predict_proba_pos >= 0.5).astype(int)
    err = 1.0 - float(accuracy_score(y, pred))
    auc = float(roc_auc_score(y, predict_proba_pos))
    return err, auc


def run_node(Xtr, ytr, Xva, yva, Xte, yte, *, trees, depth, epochs, seed, dev, batch_size=512):
    model = DenseNODE(Xtr.shape[1], num_trees=trees, depth=depth, n_layers=1)
    t0 = time.time()
    model, _ = train_node(model, Xtr, ytr, Xva, yva, lr=1e-2, max_epochs=epochs,
                          patience=max(6, epochs // 4), batch_size=batch_size, device=dev, seed=seed)
    logits = node_predict_logits(model, Xte, device=dev, batch_size=batch_size).cpu().numpy()
    p = 1.0 / (1.0 + np.exp(-np.clip(logits, -80.0, 80.0)))
    err, auc = _error_auc(p, yte)
    return err, auc, time.time() - t0


def run_catboost(Xtr, ytr, Xva, yva, Xte, yte, *, trees, seed):
    from catboost import CatBoostClassifier
    t0 = time.time()
    m = CatBoostClassifier(iterations=trees, depth=6, learning_rate=0.1, verbose=0,
                           random_seed=seed, od_type="Iter", od_wait=30)
    m.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True)
    p = m.predict_proba(Xte)[:, 1]
    err, auc = _error_auc(p, yte)
    return err, auc, time.time() - t0


def preset_cfg(name):
    if name == "smoke":
        return dict(trees=8, depth=3, epochs=2, subsample=800, batch_size=512)
    if name == "closer":
        return dict(trees=256, depth=6, epochs=25, subsample=None, batch_size=512)
    if name == "paper":
        # 2048 trees × depth 6 is the paper default. Batch 128 so the backward
        # activations fit a 16 GB T4 (Colab/Modal); Colab Pro does not add GPU RAM
        # on a T4. The dataset is still OpenML 23512 (~98k), not the paper's 10.5M.
        return dict(trees=2048, depth=6, epochs=40, subsample=None, batch_size=128)
    raise ValueError(f"unknown preset {name!r}")


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--preset", choices=("smoke", "closer", "paper"), default="closer")
    args = p.parse_args(argv)
    cfg = preset_cfg(args.preset)
    dev = device()
    hw = hardware_tag()
    print(f"L044 paper-repro  preset={args.preset}  device={hw}")

    (X, y), dname, gaps = load_table()
    if cfg["subsample"] and len(y) > cfg["subsample"]:
        rng = np.random.default_rng(0)
        idx = rng.choice(len(y), size=cfg["subsample"], replace=False)
        X, y = X[idx], y[idx]
        gaps = list(gaps) + [f"smoke subsampled to {cfg['subsample']} rows"]
    Xtr, ytr, Xva, yva, Xte, yte = _split(X, y, 0)
    print(f"  table={dname}  n={len(y)}  d={X.shape[1]}")

    node_run = None
    cb_err = None
    try:
        n_err, n_auc, n_wall = run_node(Xtr, ytr, Xva, yva, Xte, yte,
                                        trees=cfg["trees"], depth=cfg["depth"],
                                        epochs=cfg["epochs"], seed=0, dev=dev,
                                        batch_size=cfg["batch_size"])
        node_run = ScaleUpRun(
            method="node-scratch", dataset=dname, metric="error",
            value=n_err, n_seeds=1, hardware=hw, wall_s=n_wall,
            protocol_match=False,
            protocol_deviations=gaps + [
                f"trees={cfg['trees']} depth={cfg['depth']} (paper default 2048 × depth 6)",
                f"batch_size={cfg['batch_size']}",
                f"test AUC={n_auc:.4f}",
            ],
        )
        print(f"  NODE     error={n_err:.4f}  auc={n_auc:.4f}  wall={n_wall:.0f}s")
        cb_err, cb_auc, cb_wall = run_catboost(Xtr, ytr, Xva, yva, Xte, yte,
                                               trees=cfg["trees"], seed=0)
        print(f"  CatBoost error={cb_err:.4f}  auc={cb_auc:.4f}  wall={cb_wall:.0f}s")
    except Exception as exc:
        print(f"  bake-off failed: {exc}")
        traceback.print_exc()

    extra = []
    if node_run is not None and cb_err is not None:
        d = classify_direction(node_run.value, cb_err, paper_a_beats_b=True,
                               higher_is_better=False, tie_tol=0.005)
        extra.append(
            f"DIRECTION NODE vs CatBoost error on {dname}: {d}. "
            f"Paper Table 1 (default HPs, full Higgs) NODE 0.2412 vs CatBoost 0.2434 — a 0.002 edge. "
            f"A DIRECTION_TIE (or a FAIL on a toy smoke budget) means this experiment cannot see "
            f"that gap — not that Table 1 is false. Only a closer/paper-preset FAIL with a matched "
            f"protocol would be real tension with the paper's claim."
        )

    rows = [(HIGGS_TARGET, node_run, classify_number(HIGGS_TARGET, node_run))]
    text = format_ledger(title="L044 NODE", lab=LAB_FINDINGS, paper=rows, extra_lines=extra)
    print()
    print(text)

    out = {
        "lesson": 44, "preset": args.preset, "hardware": hw, "table": dname,
        "node": to_jsonable(node_run), "catboost_error": cb_err, "ledger": text,
    }
    dest = os.environ.get("PAPER_REPRO_OUT") or os.path.join(
        HERE, f"_paper_repro_l044_{args.preset}_results.json"
    )
    with open(dest, "w") as fh:
        json.dump(out, fh, indent=2)
    print(f"\nwrote {dest}")
    return out

from relkit.paper_repro import print_howto

RUN_PAPER_REPRO = False
PRESET = "closer"          # smoke | closer | paper

if RUN_PAPER_REPRO:
    main(["--preset", PRESET])
else:
    print_howto(lesson=44, modal="modal/l044_paper_repro.py", harness="labs/_paper_repro_l044.py")


## Stretch (optional, ungraded) — after the scale-up

1. **The DenseNet depth claim.** Compare `n_layers = 1` vs `2` vs `3` on one table at fixed total trees.
   NODE's pitch is that stacking lets later trees split on earlier trees' decisions — does depth help here,
   or is one layer enough at this scale?
2. **alpha as a knob.** Swap `entmax15` for `sparsemax` (alpha = 2) and softmax (alpha → 1) in the feature
   choice and re-race. The paper picks 1.5 deliberately — do you see why the extremes are worse?
3. **Give NODE its real budget.** The paper uses up to ~2000 trees per layer. Push `num_trees` up (expect
   minutes per fit on CPU) on one table and watch whether NODE closes the gap to CatBoost — and how the
   cost ratio explodes. This is the honest way to see the paper's regime.
4. **Where differentiability should win.** Concatenate two tables on a key (a tiny relational join), feed
   the joined rows to NODE, and train an embedding jointly with the trees. This is the toy version of the
   Year 4 relational setting — the case where end-to-end training is the point.

In [ ]:
# STRETCH — ungraded.
# X, y = load_dense("kc1")
# for L in (1, 2, 3):
#     f = frame(X, y, 0); torch.manual_seed(0)
#     m = DenseNODE(X.shape[1], num_trees=128, depth=4, n_layers=L)
#     m, _ = train_node(m, *f[:4], lr=1e-2, max_epochs=EPOCHS, patience=PATIENCE, seed=0)
#     print(f"n_layers={L}: test AUC {node_auc(m, f[4], f[5]):.3f}")